# LVVO 1k four-class detection EDA

**Target:** manually annotated LVVO 1k with categories (Table, Chart-Graph, Photographic-image, Visual-illustration). The single-class 1k and automatically annotated 3k archives are deliberately excluded. This notebook downloads the archive directly from the [authors’ repository](https://github.com/dbiswas55/LVVO_dataset) / [public Drive folder](https://drive.google.com/drive/folders/1RJjj71CAQHnRLqcWIYmJbWj_1zGmw2bO), verifies it, then inspects every image and label. The [paper](https://arxiv.org/abs/2506.13657) describes the annotation origin. No model is evaluated here.

**Holdout rule:** this release does not advertise a train/validation/test split. Reading all labels here does not create an unseen final test. A later benchmark needs a video-grouped split and a separately held-out test.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from io import BytesIO
import hashlib, json, re, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import gdown

SOURCE_REPO = "https://github.com/dbiswas55/LVVO_dataset"
FOLDER_URL = "https://drive.google.com/drive/folders/1RJjj71CAQHnRLqcWIYmJbWj_1zGmw2bO"
FILE_ID = "1oVufdcjhs76uDmEGu63vywKUXy5ZQSVF"
EXPECTED_NAME = "LVVO_1k_withCategories.zip"
ROOT = Path.cwd()
DATA_DIR = ROOT / "experiments" / "outputs" / "lvvo_four_class_eda" if (ROOT / "experiments").exists() else ROOT / "lvvo_four_class_eda"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE = DATA_DIR / EXPECTED_NAME
if not ARCHIVE.exists():
    downloaded = gdown.download(id=FILE_ID, output=str(ARCHIVE), quiet=False)
    if not downloaded:
        raise RuntimeError("Official LVVO four-class archive could not be downloaded; EDA is HOLD.")
archive_sha256 = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
print(pd.DataFrame([dict(source=FOLDER_URL, file_id=FILE_ID, archive=EXPECTED_NAME,
                         bytes=ARCHIVE.stat().st_size, sha256=archive_sha256)]).to_string(index=False))


## Archive integrity and safe extraction

Every member is CRC checked. Extraction rejects traversal and symlinks and writes only into the dedicated ignored output directory. The source file hash identifies this exact snapshot; the notebook does not assume a permanent hash for future upstream revisions.

In [ ]:
EXTRACT_DIR = DATA_DIR / "source"
with zipfile.ZipFile(ARCHIVE) as z:
    bad_crc = z.testzip()
    if bad_crc:
        raise RuntimeError(f"Corrupt archive member: {bad_crc}")
    members = [m for m in z.infolist() if not m.filename.startswith("__MACOSX/") and not m.filename.endswith(".DS_Store")]
    for m in members:
        target = (EXTRACT_DIR / m.filename).resolve()
        if not target.is_relative_to(EXTRACT_DIR.resolve()):
            raise RuntimeError(f"Unsafe archive path: {m.filename}")
        if (m.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError(f"Symlink in archive: {m.filename}")
    for m in members:
        target = EXTRACT_DIR / m.filename
        if m.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        elif not target.exists() or target.stat().st_size != m.file_size:
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(m) as src, target.open("wb") as dst:
                import shutil
                shutil.copyfileobj(src, dst)

dataset_root = EXTRACT_DIR / "LVVO_1k_withCategories"
image_paths = {p.stem: p for p in (dataset_root / "images").glob("*.jpg")}
label_paths = {p.stem: p for p in (dataset_root / "labels").glob("*.json")}
meta = json.loads((dataset_root / "dataset_info.json").read_text(encoding="utf-8"))
class_names = {int(v): k for k, v in meta["categories"].items()}
print("Archive members:", len(members), "images:", len(image_paths), "labels:", len(label_paths), "classes:", class_names)
print("Image without label:", len(image_paths.keys() - label_paths.keys()), "label without image:", len(label_paths.keys() - image_paths.keys()))


## Complete image and annotation audit

All image files are decoded. Counts below use explicit denominators. The filename `v` token is treated as a **candidate video grouping**, not verified against source URLs. Image dimensions in labels must match decoded pixels. Bounding boxes use the released pixel coordinate convention.

In [ ]:
rows, box_rows, errors = [], [], []
for stem in sorted(image_paths.keys() | label_paths.keys()):
    ip, lp = image_paths.get(stem), label_paths.get(stem)
    if not ip or not lp:
        errors.append(dict(stem=stem, issue="missing_pair"))
        continue
    try:
        with Image.open(ip) as im:
            im.load()
            width, height = im.size
        raw = ip.read_bytes()
        d = json.loads(lp.read_text(encoding="utf-8"))
    except Exception as exc:
        errors.append(dict(stem=stem, issue=f"read_error:{type(exc).__name__}"))
        continue
    asset = d.get("asset", {})
    size = asset.get("size", {})
    dims_match = (size.get("width"), size.get("height")) == (width, height)
    name_match = asset.get("name") == ip.name
    video_match = re.search(r"_v(\d+)_", stem)
    course_match = re.search(r"_c(\d+)_", stem)
    instructor_match = re.match(r"i(\d+)_", stem)
    rows.append(dict(stem=stem, image=ip.name, width=width, height=height,
                     image_id=asset.get("image_id"), video_id=video_match.group(1) if video_match else None,
                     course_id=course_match.group(1) if course_match else None,
                     instructor_id=instructor_match.group(1) if instructor_match else None,
                     n_objects=len(d.get("objects", [])), dimensions_match=dims_match,
                     asset_name_match=name_match, image_sha256=hashlib.sha256(raw).hexdigest()))
    for i, obj in enumerate(d.get("objects", [])):
        b = obj.get("boundingBox", {})
        x1, y1, x2, y2 = (b.get(k) for k in ("xmin", "ymin", "xmax", "ymax"))
        numeric = all(isinstance(v, (int, float)) for v in (x1, y1, x2, y2))
        valid = bool(numeric and 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height)
        box_rows.append(dict(stem=stem, object_index=i, class_id=obj.get("class"),
                             class_name=class_names.get(obj.get("class"), "UNKNOWN"),
                             xmin=x1, ymin=y1, xmax=x2, ymax=y2, valid_geometry=valid,
                             stored_width_match=numeric and b.get("width") == x2-x1,
                             stored_height_match=numeric and b.get("height") == y2-y1,
                             area_fraction=((x2-x1)*(y2-y1)/(width*height)) if valid else np.nan))

images = pd.DataFrame(rows)
boxes = pd.DataFrame(box_rows)
issues = pd.DataFrame(errors)
summary = pd.DataFrame([dict(source_images=len(image_paths), source_labels=len(label_paths),
    paired_and_decoded=len(images), missing_pair=len(image_paths.keys() ^ label_paths.keys()),
    failed_reads=len(issues) - len(image_paths.keys() ^ label_paths.keys()),
    boxes=len(boxes), invalid_boxes=int((~boxes.valid_geometry).sum()),
    dimensions_mismatch=int((~images.dimensions_match).sum()),
    asset_name_mismatch=int((~images.asset_name_match).sum()),
    unknown_class=int((boxes.class_name == "UNKNOWN").sum()),
    unique_v_tokens=images.video_id.nunique())])
display(summary)
display(issues.head(20))


## Class balance, box scale, and source grouping

In [ ]:
class_counts = boxes.groupby("class_name").agg(boxes=("stem", "size"), images=("stem", "nunique"))
class_counts["box_share"] = class_counts.boxes / len(boxes)
display(class_counts.sort_values("boxes", ascending=False))
display(images.n_objects.describe().to_frame("objects_per_image"))
display(boxes.area_fraction.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame("box_area_fraction"))
display(pd.DataFrame({"instructor_frames": images.instructor_id.value_counts(),
                      "course_frames": images.course_id.value_counts(),
                      "video_frames": images.video_id.value_counts()}).head(20))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
class_counts.boxes.sort_values().plot.barh(ax=ax[0], color="#2d77a5", title="Objects by released class")
boxes.area_fraction.dropna().plot.hist(bins=30, ax=ax[1], color="#da9944", title="Bounding-box area / image area")
ax[1].set_xlabel("Fraction of decoded image area")
plt.tight_layout(); plt.show()


## Duplicate and split leakage checks

The archive has no official split field. Exact SHA-256 duplicates and perceptual dHash candidates (Hamming distance ≤4) are checked globally. A dHash match is a **review flag**, not proof that two labels or frames are interchangeable. Cross-split leakage cannot be quantified until a split exists. A future split must group by source video (and review course/instructor relations).

In [ ]:
dupe = images.groupby("image_sha256").agg(count=("stem", "size"), examples=("stem", lambda s: list(s)[:5])).query("count > 1")
def dhash(path):
    with Image.open(path) as im:
        a = np.asarray(im.convert("L").resize((9, 8), Image.Resampling.LANCZOS))
    value = 0
    for bit in (a[:, 1:] > a[:, :-1]).ravel():
        value = (value << 1) | int(bit)
    return value

perceptual = [(r.stem, r.video_id, dhash(image_paths[r.stem])) for r in images.itertuples()]
sha_by_stem = dict(zip(images.stem, images.image_sha256))
near_pairs = []
for i, (left_stem, left_video, left_hash) in enumerate(perceptual):
    for right_stem, right_video, right_hash in perceptual[i+1:]:
        distance = (left_hash ^ right_hash).bit_count()
        if distance <= 4 and sha_by_stem[left_stem] != sha_by_stem[right_stem]:
            near_pairs.append((left_stem, right_stem, distance, left_video == right_video))
near = pd.DataFrame(near_pairs, columns=["left", "right", "dhash_distance", "same_v_token"])
display(pd.DataFrame([dict(duplicate_hash_groups=len(dupe), duplicate_images=int(dupe["count"].sum()) if len(dupe) else 0,
                           duplicate_image_ids=int(images.image_id.duplicated().sum()),
                           near_duplicate_candidate_pairs=len(near),
                           near_duplicate_cross_v_token=int((~near.same_v_token).sum()) if len(near) else 0,
                           split_column_present=False, split_leakage_status="HOLD: no released split")]))
display(dupe.head(15))
display(near.head(20))
print("Top candidate video groups:")
display(images.video_id.value_counts().head(15).rename_axis("v token").to_frame("frames"))


## Deterministic visual examples and unusual cases

One image per class is selected by sorted filename, then all released boxes are drawn. These four examples are a **sample** and cannot establish label quality for the entire corpus. Full image decoding and geometry validation above cover all files.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, name in zip(axes.ravel(), sorted(class_names.values())):
    subset = boxes[boxes.class_name == name]
    if subset.empty:
        ax.set_title(name + ": no box"); ax.axis("off"); continue
    stem = sorted(subset.stem.unique())[0]
    im = Image.open(image_paths[stem]).convert("RGB")
    draw = ImageDraw.Draw(im)
    for _, b in boxes[boxes.stem == stem].iterrows():
        if b.valid_geometry:
            draw.rectangle((b.xmin,b.ymin,b.xmax,b.ymax), outline="#ff3b30", width=5)
    ax.imshow(im); ax.set_title(f"{name}: {stem}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()
print("Smallest valid boxes:")
display(boxes[boxes.valid_geometry].nsmallest(5, "area_fraction")[["stem","class_name","area_fraction"]])
print("Largest valid boxes:")
display(boxes[boxes.valid_geometry].nlargest(5, "area_fraction")[["stem","class_name","area_fraction"]])


## Decision and reproducibility receipt

This notebook qualifies **the retrieved annotation snapshot**, subject to the printed QA issues. It does not certify copyright permissions, source-video URLs, expert annotation agreement, group-safe splits, or benchmark outcomes. The release describes manually annotated boxes for four visual categories, but an independent provenance/rights review remains needed before a formal benchmark. No untouched final test exists after this all-label EDA.

In [ ]:
blocking = int(summary.invalid_boxes.iloc[0] + summary.dimensions_mismatch.iloc[0] +
               summary.unknown_class.iloc[0] + summary.missing_pair.iloc[0] + summary.failed_reads.iloc[0])
decision = "REPAIR/HOLD" if blocking else "KEEP for four-class EDA; HOLD formal benchmark until rights, provenance and video-grouped split are verified"
receipt = dict(source_repo=SOURCE_REPO, source_folder=FOLDER_URL, file_id=FILE_ID,
               archive=EXPECTED_NAME, archive_sha256=archive_sha256, archive_bytes=ARCHIVE.stat().st_size,
               selected_images=len(image_paths), decoded_images=len(images), selected_labels=len(label_paths),
               boxes=len(boxes), class_names=class_names, blocking_qa_count=blocking, decision=decision,
               media_scope="all image files decoded", label_scope="all JSON files parsed",
               split_scope="no official split in archive; no unseen test", extra_archives_used=False)
(DATA_DIR / "eda_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
images.to_csv(DATA_DIR / "image_ledger.csv", index=False)
boxes.to_csv(DATA_DIR / "box_ledger.csv", index=False)
print(json.dumps(receipt, indent=2))
